In [ ]:
# =========================
# 1. IMPORT THƯ VIỆN
# =========================

import pandas as pd
import numpy as np
import json
import random

import tensorflow as tf

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score
)

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE


# =========================
# 2. ĐỌC DỮ LIỆU ĐÃ XỬ LÝ
# =========================

df = pd.read_csv("data_da_xu_ly.csv")

print(df.head())


# =========================
# 3. TÁCH X VÀ y
# =========================

X = df.drop("stroke", axis=1)

y = df["stroke"]


# =========================
# 4. CHIA TRAIN / TEST
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


# =========================
# 5. CÂN BẰNG DỮ LIỆU SMOTE
# =========================

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(
    X_train,
    y_train
)

print("Sau SMOTE:", X_train_res.shape)


# =========================
# 6. FEATURE ĐƯỢC CHỌN TỪ GA
# =========================
# COPY KẾT QUẢ TỪ stroke_2

selected_features_idx = [0, 2, 5]

print("Feature được chọn:", selected_features_idx)
print("Số feature:", len(selected_features_idx))


# =========================
# 7. LỌC DỮ LIỆU THEO GA
# =========================

X_train_ga = X_train_res.iloc[:, selected_features_idx]

X_test_ga = X_test.iloc[:, selected_features_idx]

print("Shape sau GA:", X_train_ga.shape)


# =====================================================
# 8. GRIDSEARCHCV - DECISION TREE
# =====================================================

dt_params = {
    'max_depth': [4, 6, 8, 10],
    'min_samples_leaf': [5, 10, 15],
    'criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

dt_grid.fit(X_train_ga, y_train_res)

best_dt = dt_grid.best_estimator_

print("\nBest Decision Tree Params:")
print(dt_grid.best_params_)


# =====================================================
# 9. GRIDSEARCHCV - SVM
# =====================================================

svm_params = {
    'C': [0.1, 1, 10],
    'gamma': [0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

svm_grid = GridSearchCV(
    SVC(
        probability=True,
        random_state=42
    ),
    svm_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

svm_grid.fit(X_train_ga, y_train_res)

best_svm = svm_grid.best_estimator_

print("\nBest SVM Params:")
print(svm_grid.best_params_)


# =====================================================
# 10. GRIDSEARCHCV - RANDOM FOREST
# =====================================================

rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10],
    'min_samples_leaf': [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

rf_grid.fit(X_train_ga, y_train_res)

best_rf = rf_grid.best_estimator_

print("\nBest Random Forest Params:")
print(rf_grid.best_params_)


# =====================================================
# 11. NAIVE BAYES
# =====================================================

nb_model = GaussianNB()

nb_model.fit(X_train_ga, y_train_res)


# =====================================================
# 12. TẠO 4 MÔ HÌNH
# =====================================================

models = {
    "Naive Bayes": nb_model,
    "Decision Tree": best_dt,
    "SVM": best_svm,
    "Random Forest": best_rf
}


# =====================================================
# 13. ĐÁNH GIÁ MÔ HÌNH
# =====================================================

model_results = {}

for name, model in models.items():

    y_probs = model.predict_proba(X_test_ga)[:, 1]

    threshold = 0.45

    y_pred = (y_probs > threshold).astype(int)

    auc = roc_auc_score(y_test, y_probs)

    print("\n================================================")
    print(name)
    print("================================================")

    print(classification_report(y_test, y_pred))

    print("ROC-AUC:", auc)

    model_results[name] = {
        "ROC_AUC": float(auc)
    }


# =====================================================
# 14. NEURAL NETWORK
# =====================================================

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

nn_model = tf.keras.Sequential([

    tf.keras.layers.Dense(
        32,
        activation='relu',
        input_shape=(len(selected_features_idx),)
    ),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        16,
        activation='relu'
    ),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(
        1,
        activation='sigmoid'
    )
])

nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['AUC']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)

nn_model.fit(
    X_train_ga,
    y_train_res,
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop]
)

nn_probs = nn_model.predict(X_test_ga)

nn_preds = (nn_probs > 0.45).astype(int)

nn_auc = roc_auc_score(y_test, nn_probs)

print("\n================================================")
print("Neural Network")
print("================================================")

print(classification_report(y_test, nn_preds))

print("ROC-AUC:", nn_auc)

model_results["Neural Network"] = {
    "ROC_AUC": float(nn_auc)
}


# =====================================================
# 15. LƯU THÔNG SỐ QUAN TRỌNG
# =====================================================

important_parameters = {

    "Selected Features": selected_features_idx,

    "Decision Tree Best Params":
        dt_grid.best_params_,

    "SVM Best Params":
        svm_grid.best_params_,

    "Random Forest Best Params":
        rf_grid.best_params_,

    "Model Results":
        model_results
}


# =====================================================
# 16. LƯU FILE JSON
# =====================================================

with open("model_parameters.json", "w") as f:

    json.dump(
        important_parameters,
        f,
        indent=4
    )

print("\nĐã lưu model_parameters.json")

          0         1         2         3         4         5    6         7  \
0  1.087508 -0.289936 -0.219621 -0.110351 -0.142117  0.932544  0.0  0.487671   
1  0.315465 -0.289936 -0.219621 -0.559893  1.026223 -1.072336  0.0  0.487671   
2 -1.546524 -0.289936 -0.219621 -1.092245  0.636776 -1.072336  0.0 -2.050562   
3 -1.092380 -0.289936 -0.219621 -0.978011  0.142478  0.932544  0.0  0.487671   
4 -1.092380 -0.289936 -0.219621 -1.090212 -1.430288 -1.072336  0.0 -2.050562   

          8         9        10        11        12        13        14  \
0 -0.050695 -1.494878  1.549053 -0.282598 -1.342176  1.487287 -0.923454   
1 -0.050695  0.668951 -0.645556 -0.282598  0.745059 -0.672365 -0.923454   
2 -0.050695  0.668951 -0.645556 -0.282598 -1.342176 -0.672365  1.082891   
3 -0.050695 -1.494878 -0.645556 -0.282598  0.745059  1.487287 -0.923454   
4 -0.050695  0.668951 -0.645556 -0.282598 -1.342176 -0.672365  1.082891   

         15  stroke  
0 -0.548818       0  
1 -0.548818       0  
2 

d:\IT\HocMay\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - AUC: 0.8197 - loss: 0.5298
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.8525 - loss: 0.4731
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.8579 - loss: 0.4646
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.8589 - loss: 0.4636
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.8591 - loss: 0.4620
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.8606 - loss: 0.4603
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.8625 - loss: 0.4590
Epoch 8/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.8604 - loss: 0.4622
Epoch 9/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.8627 - loss: 0.4574
Epoch 10/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.8631 - loss: 0.4589
Epoch 11/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.8631 - loss: 0.4569
Epoch 12/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - AUC: 0.8628 - loss: 0.4568
Epoch 13/50
196/196 ━━━━━